# ETL source: electricity Silver

Add this notebook AND `03_gold_electricity_etl.ipynb` to ONE ETL pipeline.
Default catalog/schema: **db_labb2 / etl_demo**. Session time zone: **UTC**.
Run via the pipeline, not an ordinary Notebook task.

Output: `db_labb2.etl_demo.silver_electricity_prices` (materialized view).
The original nine business columns and units are retained. `_key_count` is an
additional quality-check column; it must always be 1.
Validation covers missing/duplicate day-area responses, timestamps with offsets,
numeric types and finite values, complete local days, gaps/overlaps, and duplicate
interval keys. Negative prices are allowed.

Unlike the original Silver notebook, the count of DST corrections is not required
to be exactly four: a shorter sample can exclude that date. The same narrowly
defined correction is applied whenever the erroneous source interval occurs.
This version uses a Python UDF to preserve the original validation semantics;
do not assume every refresh will be incremental or cheaper than the original job.


In [ ]:
import json
import math
from datetime import date, datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo

from pyspark import pipelines as dp
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, DateType, DoubleType, StringType, StructField,
    StructType, TimestampType,
)

# Set the pipeline default catalog/schema to db_labb2 / etl_demo.
# Optional pipeline configuration overrides:
# electricity.bronze_table = db_labb2.bronze.electricity_price_raw
# electricity.start_date = 2025-01-01
BRONZE_TABLE = spark.conf.get(
    "electricity.bronze_table", "db_labb2.bronze.electricity_price_raw"
)
PRICE_START_TEXT = spark.conf.get("electricity.start_date", "2025-01-01")
PRICE_START = date.fromisoformat(PRICE_START_TEXT)
AREAS = ("SE1", "SE2", "SE3", "SE4")
STOCKHOLM = ZoneInfo("Europe/Stockholm")

PRICE_SCHEMA = StructType([
    StructField("price_area", StringType(), False),
    StructField("price_date", DateType(), False),
    StructField("time_start", TimestampType(), False),
    StructField("time_end", TimestampType(), False),
    StructField("sek_per_kwh", DoubleType(), False),
    StructField("eur_per_kwh", DoubleType(), False),
    StructField("exchange_rate", DoubleType(), False),
    StructField("duration_minutes", DoubleType(), False),
    StructField("source_time_end", TimestampType(), False),
])


In [ ]:
def utc_timestamp(value):
    parsed = datetime.fromisoformat(value)
    if parsed.tzinfo is None or parsed.utcoffset() is None:
        raise ValueError(f"Electricity timestamp has no UTC offset: {value}")
    return parsed.astimezone(timezone.utc)


def parse_price_response(payload_json, source, day, area,
                         response_count, expected_key):
    """Pure transformation: validate one day/area, return its interval rows.

    No API requests, Spark actions, table writes or mutable external state.
    Error messages identify the failing day and price area.
    """
    context = f"Electricity Silver; date={day}; area={area}"
    try:
        if (expected_key != 1 or response_count != 1
                or source != "Elpriset just nu" or area not in AREAS
                or day is None or day < PRICE_START):
            raise ValueError(
                f"missing/unexpected/duplicate Bronze response; "
                f"source={source}; responses={response_count}; expected={expected_key}"
            )
        payload = json.loads(payload_json)
        if not isinstance(payload, list) or not payload:
            raise ValueError("payload must be a non-empty JSON array")
        minutes = 60 if day < date(2025, 10, 1) else 15
        intervals, rows = [], []
        for item in payload:
            start = utc_timestamp(item["time_start"])
            raw_end = utc_timestamp(item["time_end"])
            end = start + timedelta(minutes=minutes)
            if start.astimezone(STOCKHOLM).date() != day:
                raise ValueError(f"interval belongs to another local date: {start}")
            if raw_end != end:
                # Preserve the original, narrowly scoped 2025 DST correction.
                known_error = (
                    day == date(2025, 10, 26)
                    and start == datetime(2025, 10, 26, 0, 45, tzinfo=timezone.utc)
                    and raw_end == datetime(2025, 10, 26, 2, 0, tzinfo=timezone.utc)
                    and end == datetime(2025, 10, 26, 1, 0, tzinfo=timezone.utc)
                )
                if not known_error:
                    raise ValueError(f"unexpected interval end: {item}")
            amounts = (item["SEK_per_kWh"], item["EUR_per_kWh"], item["EXR"])
            if any(isinstance(v, bool) or not isinstance(v, (int, float))
                   or not math.isfinite(v) for v in amounts):
                raise ValueError(f"invalid price/rate: {item}")
            intervals.append((start, end))
            rows.append((area, day, start, end, float(amounts[0]),
                         float(amounts[1]), float(amounts[2]),
                         float(minutes), raw_end))
        intervals.sort()
        first = datetime.combine(day, time.min, STOCKHOLM).astimezone(timezone.utc)
        last = datetime.combine(day + timedelta(days=1), time.min,
                                STOCKHOLM).astimezone(timezone.utc)
        if (intervals[0][0] != first or intervals[-1][1] != last
                or len(intervals) != (last-first).total_seconds()/(minutes*60)
                or any(a[1] != b[0] for a, b in zip(intervals, intervals[1:]))):
            raise ValueError("interval count/boundaries invalid, or gap/overlap")
        return rows
    except (ValueError, KeyError, TypeError, OverflowError) as exc:
        raise ValueError(f"{context}: {exc}") from exc


parse_price_udf = F.udf(parse_price_response, ArrayType(PRICE_SCHEMA, False))


In [ ]:
@dp.materialized_view(
    name="silver_electricity_prices",
    comment="Validated UTC electricity intervals; prices in SEK/kWh and EUR/kWh",
)
@dp.expect_or_fail("unique_interval_key", "_key_count = 1")
@dp.expect_or_fail("valid_interval", "time_start IS NOT NULL AND time_end > time_start")
def silver_electricity_prices():
    raw = spark.read.table(BRONZE_TABLE).select(
        "source", "price_date", "price_area", "payload_json"
    ).withColumn("_response_present", F.lit(1))

    # Check ALL days/areas from the configured start to the latest stored date.
    # Missing whole days must not disappear silently in a GROUP BY.
    bounds = raw.agg(F.max("price_date").alias("_last_day"))
    dates = bounds.select(F.explode_outer(
        F.when(F.col("_last_day") >= F.lit(PRICE_START),
               F.sequence(F.lit(PRICE_START), F.col("_last_day")))
    ).alias("price_date"))
    areas = spark.range(4).select(
        F.concat(F.lit("SE"), (F.col("id")+1).cast("string")).alias("price_area")
    )
    expected = dates.crossJoin(areas).withColumn("_expected_key", F.lit(1))
    checked = expected.join(raw, ["price_date", "price_area"], "full_outer")
    checked = checked.withColumn(
        "_response_count",
        F.count("_response_present").over(Window.partitionBy("price_date", "price_area")),
    )
    parsed = checked.select(F.explode(parse_price_udf(
        "payload_json", "source", "price_date", "price_area",
        "_response_count", "_expected_key",
    )).alias("interval")).select("interval.*")
    return parsed.withColumn(
        "_key_count", F.count(F.lit(1)).over(Window.partitionBy("price_area", "time_start"))
    )
